# Fashion MNIST — OpenVINO Performance Evaluator

This notebook is a **completely self-contained** walkthrough that:

1. **Trains** a small CNN on Fashion MNIST with PyTorch
2. **Converts** directly to OpenVINO Intermediate Representation (IR) — no ONNX step needed
3. **Quantizes** to INT8 (post-training) and INT4 (weight compression) with NNCF
4. **Benchmarks** every variant on every available device (CPU, GPU if present)
5. **Visualizes** the results so you can evaluate the value of OpenVINO at a glance

> **No imports from the `src/` package are used** — everything is defined inline.

## 1. Import Required Libraries and Setup

In [1]:
from __future__ import annotations

import json
import os
import platform
import re
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import nncf
import numpy as np
import openvino as ov
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ── Paths (relative to this notebook) ──────────────────────────────────────────
ROOT_DIR: Path = Path.cwd().parent          # repo root
DATA_DIR: Path = ROOT_DIR / "data"
MODELS_DIR: Path = ROOT_DIR / "models"
OPENVINO_DIR: Path = MODELS_DIR / "openvino"
BENCHMARK_LOGS_DIR: Path = ROOT_DIR / "benchmark_logs"
PYTORCH_MODEL_PATH: Path = MODELS_DIR / "fashion_mnist_cnn.pth"

# Create directories
for d in [DATA_DIR, MODELS_DIR, OPENVINO_DIR, BENCHMARK_LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────────
BATCH_SIZE: int = 64
LEARNING_RATE: float = 1e-3
EPOCHS: int = 10
NUM_CLASSES: int = 10
INPUT_SHAPE: tuple[int, int, int, int] = (1, 1, 28, 28)  # NCHW

CLASS_NAMES: list[str] = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

QUANTIZATION_VARIANTS: list[str] = ["fp32", "int8", "int4"]

print(f"PyTorch  : {torch.__version__}")
print(f"OpenVINO : {ov.__version__}")
print(f"NNCF     : {nncf.__version__}")
print(f"Root dir : {ROOT_DIR}")

PyTorch  : 2.10.0
OpenVINO : 2025.4.1-20426-82bbf0292c5-releases/2025/4
NNCF     : 2.19.0
Root dir : /Users/rashedtalukder/Library/CloudStorage/OneDrive-Personal/Engineering/Python3/github/rashedtalukder/fashion-bench


## 2. Load and Prepare Fashion MNIST Dataset

Download the Fashion MNIST dataset, apply normalization transforms, and create data loaders. We'll also visualize a grid of sample images with their class labels.

In [ ]:
# ── Transforms ─────────────────────────────────────────────────────────────────
_transform: transforms.Compose = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

# ── Datasets ───────────────────────────────────────────────────────────────────
train_dataset: datasets.FashionMNIST = datasets.FashionMNIST(
    root=str(DATA_DIR), train=True, download=True, transform=_transform,
)
test_dataset: datasets.FashionMNIST = datasets.FashionMNIST(
    root=str(DATA_DIR), train=False, download=True, transform=_transform,
)

# ── Data loaders ───────────────────────────────────────────────────────────────
train_loader: DataLoader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader: DataLoader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples : {len(train_dataset):,}")
print(f"Test samples     : {len(test_dataset):,}")

# ── Visualize sample images ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img: torch.Tensor = train_dataset[i][0].squeeze()
    label: int = train_dataset[i][1]
    ax.imshow(img * 0.5 + 0.5, cmap="gray")  # undo normalization
    ax.set_title(CLASS_NAMES[label], fontsize=10)
    ax.axis("off")
fig.suptitle("Fashion MNIST — Sample Images", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Define the CNN Model Architecture

A lightweight CNN with three conv blocks followed by adaptive average pooling and a two-layer classifier. This architecture is intentionally small to keep training fast while still achieving ~88–90 % accuracy on Fashion MNIST.

In [ ]:
class FashionCNN(nn.Module):
    """A small CNN suitable for Fashion-MNIST (28×28 grayscale images)."""

    def __init__(self) -> None:
        super().__init__()
        self.features: nn.Sequential = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier: nn.Sequential = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


model: FashionCNN = FashionCNN()
print(model)
total_params: int = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 4. Train the CNN Model

Train using **CrossEntropyLoss** and **Adam** optimizer for the configured number of epochs. If a previously trained model already exists, it is loaded instead to save time. Set `FORCE_TRAIN = True` below to retrain from scratch.

In [ ]:
FORCE_TRAIN: bool = False

device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")

if not FORCE_TRAIN and PYTORCH_MODEL_PATH.exists():
    print(f"Trained model found at {PYTORCH_MODEL_PATH} — loading weights.")
    model.load_state_dict(torch.load(PYTORCH_MODEL_PATH, map_location="cpu", weights_only=True))
    model.eval()
else:
    model.to(device)
    criterion: nn.CrossEntropyLoss = nn.CrossEntropyLoss()
    optimizer: optim.Adam = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    model.train()
    for epoch in range(1, EPOCHS + 1):
        running_loss: float = 0.0
        correct: int = 0
        total: int = 0
        t0: float = time.time()

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs: torch.Tensor = model(images)
            loss: torch.Tensor = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += int(predicted.eq(labels).sum().item())

        epoch_loss: float = running_loss / total
        epoch_acc: float = 100.0 * correct / total
        elapsed: float = time.time() - t0
        print(
            f"  Epoch {epoch:>2}/{EPOCHS}  —  "
            f"loss: {epoch_loss:.4f}  acc: {epoch_acc:.2f}%  "
            f"({elapsed:.1f}s)"
        )

    # Save the model
    PYTORCH_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    model.cpu().eval()
    torch.save(model.state_dict(), PYTORCH_MODEL_PATH)
    print(f"\nModel saved → {PYTORCH_MODEL_PATH}")

## 5. Evaluate the Trained PyTorch Model

Run the test set through the PyTorch model to establish a **baseline accuracy** before any OpenVINO conversions or quantization.

In [ ]:
model.eval()
correct: int = 0
total: int = 0
per_class_correct: list[int] = [0] * NUM_CLASSES
per_class_total: list[int] = [0] * NUM_CLASSES

with torch.no_grad():
    for images, labels in test_loader:
        outputs: torch.Tensor = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += int(predicted.eq(labels).sum().item())
        for i in range(labels.size(0)):
            label: int = int(labels[i])
            per_class_total[label] += 1
            if predicted[i] == label:
                per_class_correct[label] += 1

pytorch_accuracy: float = 100.0 * correct / total
print(f"PyTorch test accuracy: {pytorch_accuracy:.2f}%\n")

print("Per-class accuracy:")
for idx in range(NUM_CLASSES):
    acc: float = 100.0 * per_class_correct[idx] / per_class_total[idx] if per_class_total[idx] else 0.0
    print(f"  {CLASS_NAMES[idx]:<14s}: {acc:.2f}%")

## 6. Convert PyTorch Model to OpenVINO Intermediate Representation (IR)

Use `ov.convert_model()` to convert the PyTorch `nn.Module` **directly** to OpenVINO IR — no ONNX export needed. Passing `example_input` triggers `torch.jit.trace` internally and produces a higher-quality IR.

> See: [Converting a PyTorch Model — OpenVINO docs](https://docs.openvino.ai/2025/openvino-workflow/model-preparation/convert-model-pytorch.html)

In [ ]:
fp32_dir: Path = OPENVINO_DIR / "fp32"
fp32_dir.mkdir(parents=True, exist_ok=True)

# Convert PyTorch model directly to OpenVINO IR (FP32)
example_input: torch.Tensor = torch.randn(*INPUT_SHAPE)
ov_model: ov.Model = ov.convert_model(model, example_input=example_input)

fp32_xml: Path = fp32_dir / "fashion_mnist.xml"
ov.save_model(ov_model, str(fp32_xml))

print(f"OpenVINO FP32 IR saved → {fp32_xml}")
print(f"XML size: {fp32_xml.stat().st_size / 1024:.1f} KB")
bin_path: Path = fp32_xml.with_suffix(".bin")
print(f"BIN size: {bin_path.stat().st_size / 1024:.1f} KB")

## 8. Quantize Model to INT8 Using NNCF

Use **NNCF post-training quantization** (`nncf.quantize`) to produce an INT8 model. A small calibration dataset from the test set is used to calibrate activation ranges.

In [ ]:
def _calibration_data(loader: DataLoader, num_samples: int = 300) -> list[np.ndarray]:
    """Collect numpy arrays from the data loader for NNCF calibration."""
    samples: list[np.ndarray] = []
    for images, _ in loader:
        for img in images:
            if len(samples) >= num_samples:
                return samples
            samples.append(img.unsqueeze(0).numpy())
    return samples


# Read the FP32 model
core: ov.Core = ov.Core()
fp32_ov_model: ov.Model = core.read_model(str(fp32_xml))

# Build calibration dataset
cal_loader: DataLoader = DataLoader(test_dataset, batch_size=1, shuffle=False)
calibration_dataset: nncf.Dataset = nncf.Dataset(_calibration_data(cal_loader))

# Quantize to INT8
int8_model: ov.Model = nncf.quantize(
    fp32_ov_model,
    calibration_dataset,
    model_type=nncf.ModelType.TRANSFORMER,
    preset=nncf.QuantizationPreset.PERFORMANCE,
)

int8_dir: Path = OPENVINO_DIR / "int8"
int8_dir.mkdir(parents=True, exist_ok=True)
int8_xml: Path = int8_dir / "fashion_mnist.xml"
ov.save_model(int8_model, str(int8_xml))
print(f"INT8 IR saved → {int8_xml}")
print(f"BIN size: {int8_xml.with_suffix('.bin').stat().st_size / 1024:.1f} KB")

## 9. Quantize Model to INT4 Using NNCF

Use **NNCF weight compression** (`nncf.compress_weights`) with `INT4_SYM` mode to produce an INT4 model. This gives the most aggressive size reduction at the cost of some accuracy.

In [ ]:
# Read FP32 model again for INT4 compression
fp32_ov_model_for_int4: ov.Model = core.read_model(str(fp32_xml))

int4_model: ov.Model = nncf.compress_weights(
    fp32_ov_model_for_int4,
    mode=nncf.CompressWeightsMode.INT4_SYM,
)

int4_dir: Path = OPENVINO_DIR / "int4"
int4_dir.mkdir(parents=True, exist_ok=True)
int4_xml: Path = int4_dir / "fashion_mnist.xml"
ov.save_model(int4_model, str(int4_xml))
print(f"INT4 IR saved → {int4_xml}")
print(f"BIN size: {int4_xml.with_suffix('.bin').stat().st_size / 1024:.1f} KB")

## 10. Detect Available Hardware (CPU / GPU)

Query the OpenVINO runtime for available devices. We always benchmark on **CPU**. If a GPU is available, we also benchmark on **GPU**.

In [ ]:
all_devices: list[str] = core.available_devices
print(f"All OpenVINO devices: {all_devices}")

# Keep only CPU and GPU for benchmarking
benchmark_devices: list[str] = [d for d in all_devices if d in ("CPU", "GPU")]
if not benchmark_devices:
    benchmark_devices = ["CPU"]

print(f"Benchmark devices : {benchmark_devices}")


def _processor_tag() -> str:
    """Return a short, filesystem-safe tag derived from the processor name."""
    raw: str = platform.processor()
    if platform.system() == "Darwin":
        try:
            chip: str = subprocess.check_output(
                ["sysctl", "-n", "machdep.cpu.brand_string"], timeout=2,
            ).decode().strip()
            if chip:
                raw = chip
        except Exception:
            pass
    if platform.system() == "Linux" and (not raw or raw == platform.machine()):
        try:
            with open("/proc/cpuinfo", "r", encoding="utf-8") as f:
                for line in f:
                    if line.startswith("model name"):
                        raw = line.split(":", 1)[1].strip()
                        break
                    if line.startswith("Hardware"):
                        raw = line.split(":", 1)[1].strip()
        except Exception:
            pass
    if not raw:
        raw = platform.machine() or "unknown"
    tag: str = re.sub(r"[^a-z0-9]+", "_", raw.lower()).strip("_")
    tag = tag.replace("_r_", "_").replace("_tm_", "_").replace("_cpu", "")
    tag = re.sub(r"_+", "_", tag)
    if len(tag) > 48:
        tag = tag[:48].rstrip("_")
    return tag or "unknown"


def _processor_name() -> str:
    """Return a human-readable processor name."""
    raw: str = platform.processor()
    if platform.system() == "Darwin":
        try:
            chip: str = subprocess.check_output(
                ["sysctl", "-n", "machdep.cpu.brand_string"], timeout=2,
            ).decode().strip()
            if chip:
                raw = chip
        except Exception:
            pass
    if platform.system() == "Linux" and (not raw or raw == platform.machine()):
        try:
            with open("/proc/cpuinfo", "r", encoding="utf-8") as f:
                for line in f:
                    if line.startswith("model name"):
                        raw = line.split(":", 1)[1].strip()
                        break
                    if line.startswith("Hardware"):
                        raw = line.split(":", 1)[1].strip()
        except Exception:
            pass
    return (raw or platform.machine() or "Unknown").strip()


def _system_info() -> dict[str, str]:
    """Collect basic system information for the benchmark log."""
    return {
        "system": platform.system(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "python_version": platform.python_version(),
        "hostname": platform.node(),
    }


proc_tag: str = _processor_tag()
proc_name: str = _processor_name()
print(f"Processor tag  : {proc_tag}")
print(f"Processor name : {proc_name}")

## 11–16. Benchmark All Model Variants on All Devices

Run inference on the full test set for every combination of **model variant** (FP32, INT8, INT4) and **device** (CPU, GPU if available). For each run we measure:

| Metric | Description |
|---|---|
| `accuracy_pct` | Percentage of correctly classified images |
| `avg_latency_ms` | Average per-image inference latency (ms) |
| `total_time_s` | Wall-clock time for the entire test set |
| `per_class_accuracy` | Accuracy broken down by Fashion MNIST class |

In [ ]:
BenchmarkResult = dict[str, Any]


def benchmark_model(
    xml_path: Path,
    device: str = "CPU",
    variant_name: str = "fp32",
) -> BenchmarkResult:
    """Run the full test set through an OpenVINO model and collect metrics."""
    compiled_model: ov.CompiledModel = core.compile_model(str(xml_path), device)
    infer_request: ov.InferRequest = compiled_model.create_infer_request()

    bench_loader: DataLoader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    correct: int = 0
    total: int = 0
    latencies: list[float] = []
    per_cls_correct: list[int] = [0] * NUM_CLASSES
    per_cls_total: list[int] = [0] * NUM_CLASSES

    overall_start: float = time.perf_counter()

    for images, labels in bench_loader:
        img_np: np.ndarray = images.numpy()
        label: int = int(labels.item())

        t0: float = time.perf_counter()
        infer_request.infer({0: img_np})
        t1: float = time.perf_counter()

        output: np.ndarray = infer_request.get_output_tensor(0).data
        predicted: int = int(np.argmax(output, axis=1)[0])

        latencies.append((t1 - t0) * 1000.0)
        total += 1
        per_cls_total[label] += 1
        if predicted == label:
            correct += 1
            per_cls_correct[label] += 1

    overall_end: float = time.perf_counter()

    accuracy: float = 100.0 * correct / total if total > 0 else 0.0
    avg_latency_ms: float = float(np.mean(latencies)) if latencies else 0.0
    total_time_s: float = overall_end - overall_start

    per_class_accuracy: dict[str, float] = {}
    for idx in range(NUM_CLASSES):
        if per_cls_total[idx] > 0:
            per_class_accuracy[CLASS_NAMES[idx]] = round(
                100.0 * per_cls_correct[idx] / per_cls_total[idx], 2
            )
        else:
            per_class_accuracy[CLASS_NAMES[idx]] = 0.0

    return {
        "variant": variant_name,
        "device": device,
        "processor_name": proc_name,
        "model_path": str(xml_path),
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "system_info": _system_info(),
        "total_images": total,
        "accuracy_pct": round(accuracy, 2),
        "avg_latency_ms": round(avg_latency_ms, 4),
        "total_time_s": round(total_time_s, 3),
        "per_class_accuracy": per_class_accuracy,
    }


print("Benchmark function defined ✓")

In [ ]:
# ── Run benchmarks for every variant × device ─────────────────────────────────
model_variants: dict[str, Path] = {
    "fp32": fp32_xml,
    "int8": int8_xml,
    "int4": int4_xml,
}

all_results: list[BenchmarkResult] = []

for variant, xml_path in model_variants.items():
    if not xml_path.exists():
        print(f"Skipping {variant} — IR not found at {xml_path}")
        continue
    for dev in benchmark_devices:
        print(f"\n{'─'*50}")
        print(f"Benchmarking {variant.upper()} on {dev} …")
        try:
            result: BenchmarkResult = benchmark_model(xml_path, device=dev, variant_name=variant)
        except RuntimeError as exc:
            print(f"  FAILED: {exc}")
            continue

        print(f"  Accuracy    : {result['accuracy_pct']:.2f}%")
        print(f"  Avg latency : {result['avg_latency_ms']:.4f} ms")
        print(f"  Total time  : {result['total_time_s']:.3f} s")

        all_results.append(result)

print(f"\n{'═'*50}")
print(f"Completed {len(all_results)} benchmark run(s).")

## 17. Collect and Store Benchmark Results

Save individual JSON logs per variant/device combination, a combined log, and update the manifest file so the HTML viewer can discover them.

In [ ]:
BENCHMARK_LOGS_DIR.mkdir(parents=True, exist_ok=True)

# ── Write individual JSON logs ─────────────────────────────────────────────────
for result in all_results:
    log_name: str = f"{result['variant']}_{result['device'].lower()}_{proc_tag}.json"
    log_path: Path = BENCHMARK_LOGS_DIR / log_name
    with open(log_path, "w", encoding="utf-8") as fh:
        json.dump(result, fh, indent=2)
    print(f"  {log_name}")

# ── Write combined log ─────────────────────────────────────────────────────────
combined_path: Path = BENCHMARK_LOGS_DIR / "all_benchmarks.json"
with open(combined_path, "w", encoding="utf-8") as fh:
    json.dump(all_results, fh, indent=2)
print(f"\nCombined log → {combined_path}")

# ── Write manifest ─────────────────────────────────────────────────────────────
log_files: list[str] = sorted(
    f.name for f in BENCHMARK_LOGS_DIR.glob("*.json") if f.name != "manifest.json"
)
manifest_path: Path = BENCHMARK_LOGS_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as fh:
    json.dump({"files": log_files}, fh, indent=2)
print(f"Manifest → {manifest_path}")

## 18. Visualize Benchmark Comparisons

Compare all benchmark runs across model variants and devices with bar charts for:

1. **Accuracy** — does quantization degrade classification quality?
2. **Average latency** — how much faster is each variant?
3. **Throughput (FPS)** — images per second
4. **Speedup** — relative to the FP32 CPU baseline

In [ ]:
if not all_results:
    print("No benchmark results to visualize.")
else:
    labels: list[str] = [f"{r['variant'].upper()}\n{r['device']}" for r in all_results]
    accuracies: list[float] = [r["accuracy_pct"] for r in all_results]
    avg_latencies: list[float] = [r["avg_latency_ms"] for r in all_results]
    total_times: list[float] = [r["total_time_s"] for r in all_results]
    fps_values: list[float] = [r["total_images"] / r["total_time_s"] if r["total_time_s"] > 0 else 0 for r in all_results]

    # Compute speedup relative to FP32 CPU baseline
    baseline_latency: float = next(
        (r["avg_latency_ms"] for r in all_results if r["variant"] == "fp32" and r["device"] == "CPU"),
        avg_latencies[0],
    )
    speedups: list[float] = [baseline_latency / lat if lat > 0 else 0.0 for lat in avg_latencies]

    # Color map: FP32=blue, INT8=green, INT4=orange
    color_map: dict[str, str] = {"fp32": "#4285F4", "int8": "#34A853", "int4": "#FA7B17"}
    colors: list[str] = [color_map.get(r["variant"], "#888888") for r in all_results]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1) Accuracy
    ax = axes[0, 0]
    bars = ax.bar(labels, accuracies, color=colors)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Test Accuracy")
    ax.set_ylim(max(0, min(accuracies) - 5), 100)
    for bar, val in zip(bars, accuracies):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3, f"{val:.1f}%",
                ha="center", va="bottom", fontsize=9)

    # 2) Average Latency
    ax = axes[0, 1]
    bars = ax.bar(labels, avg_latencies, color=colors)
    ax.set_ylabel("Avg Latency (ms)")
    ax.set_title("Average Per-Image Latency")
    for bar, val in zip(bars, avg_latencies):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002, f"{val:.3f}",
                ha="center", va="bottom", fontsize=9)

    # 3) Throughput (FPS)
    ax = axes[1, 0]
    bars = ax.bar(labels, fps_values, color=colors)
    ax.set_ylabel("Images / second")
    ax.set_title("Throughput (FPS)")
    for bar, val in zip(bars, fps_values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10, f"{val:.0f}",
                ha="center", va="bottom", fontsize=9)

    # 4) Speedup relative to FP32 CPU
    ax = axes[1, 1]
    bars = ax.bar(labels, speedups, color=colors)
    ax.set_ylabel("Speedup (×)")
    ax.set_title("Speedup vs FP32 CPU Baseline")
    ax.axhline(y=1.0, color="gray", linestyle="--", linewidth=0.8)
    for bar, val in zip(bars, speedups):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{val:.2f}×",
                ha="center", va="bottom", fontsize=9)

    fig.suptitle(f"OpenVINO Benchmark — {proc_name}", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # ── Summary table ──────────────────────────────────────────────────────────
    print(f"\n{'Variant':<8} {'Device':<6} {'Accuracy':>10} {'Latency (ms)':>14} {'FPS':>8} {'Speedup':>9}")
    print("─" * 58)
    for r, spd in zip(all_results, speedups):
        fps: float = r["total_images"] / r["total_time_s"] if r["total_time_s"] > 0 else 0
        print(
            f"{r['variant'].upper():<8} {r['device']:<6} "
            f"{r['accuracy_pct']:>9.2f}% "
            f"{r['avg_latency_ms']:>13.4f} "
            f"{fps:>8.0f} "
            f"{spd:>8.2f}×"
        )

: 

## Per-Class Accuracy Comparison

Visualize how quantization affects accuracy for each of the 10 Fashion MNIST classes.

In [ ]:
if not all_results:
    print("No benchmark results to visualize.")
else:
    # Only plot CPU results for per-class comparison to avoid clutter
    cpu_results: list[BenchmarkResult] = [r for r in all_results if r["device"] == "CPU"]
    if not cpu_results:
        cpu_results = all_results

    x_pos: np.ndarray = np.arange(NUM_CLASSES)
    width: float = 0.8 / len(cpu_results)

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, r in enumerate(cpu_results):
        per_cls: dict[str, float] = r["per_class_accuracy"]
        values: list[float] = [per_cls.get(name, 0.0) for name in CLASS_NAMES]
        offset: float = (i - len(cpu_results) / 2 + 0.5) * width
        ax.bar(x_pos + offset, values, width, label=f"{r['variant'].upper()} ({r['device']})",
               color=color_map.get(r["variant"], "#888"))

    ax.set_xticks(x_pos)
    ax.set_xticklabels(CLASS_NAMES, rotation=35, ha="right")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"Per-Class Accuracy — {proc_name}")
    ax.set_ylim(50, 100)
    ax.legend()
    plt.tight_layout()
    plt.show()